In [1]:
import requests 
from bs4 import BeautifulSoup as bs
import pandas as pd
import tldextract
import numpy as np
import matplotlib.pylab as plt
import os
import re

In [2]:
def on_sale_chk(text):
    if len(text)<1:
        return False
    return 'domain' in text and 'sale' in text

def on_parked_chk(text):
    if len(text)<1:
        return True
    return 'domain' in text and 'park' in text

def on_Parked(text):
    if len(text)<1:
        return True
    return (('website' in text or 'content' in text) and 'unavailable' in text) or ('will' in text and 'soon' in text)

In [3]:
#returns html contents, textual character length, website size, status code, parked or on sale

def soupFromUrl(scrapeUrl):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
    try:
        req = requests.get(scrapeUrl, headers=headers, timeout=5)
        # print(req.status_code)
        req.close()
        if req.status_code == 200:
            # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
            soup = bs(req.text,'lxml')

            # print(soup)

            text = ''

            if soup.body:
                text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

            # print(soup)

            # print('text',text)
            # print(bs(req.text, 'html.parser'))
            # return [bs(req.text, 'html.parser'),len(req.text), len(req.content), req.status_code]
            # print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])
            return [len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))]
        else:
            # return [-1,0,0,req.status_code]
            return [0,0,req.status_code,0]
    except:
        # return [-1,0,0,-1]
        return [0,0,-1,0]

In [4]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
req = requests.get('https://www.lycos.com/', headers=headers, timeout=5)
print(req.status_code)
req.close()
if req.status_code == 200:
    # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
    soup = bs(req.text,'lxml')

    # print(soup)
    text = ''

    if soup.body:
        text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

    print(soup)
    print('text',text)
    print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])

200
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<!-- The above 3 meta tags *must* come first in the head; any other head content must come *after* these tags -->
<meta content="Lycos, Inc., is a web search engine and web portal established in 1994, spun out of Carnegie Mellon University. Lycos also encompasses a network of email, webhosting, social networking, and entertainment websites." name="description"/>
<meta content="" name="author"/>
<link href="https://ly.lygo.net/static/lycos/img/favicon.ico" rel="icon" type="image/png"/>
<title>Lycos.com</title>
<link href="//fonts.googleapis.com/css?family=Lato:400,300,300italic,400italic,700,700italic" rel="stylesheet" type="text/css"/>
<link href="/css/in/fonts.css" rel="stylesheet" type="text/css"/>
<link href="https://ly.lygo.net/static/lycos/css/in/font-awesome.css" rel="stylesheet" type="text

In [5]:
# print(soupFromUrl('https://www.delinian.com/'))
# print(soupFromUrl('https://www.makecashonline.com/'))
print(soupFromUrl('https://www.lycos.com/'))

[13587, 328, 200, 0]


In [8]:
swahili_data_url = list(pd.read_csv('../URL Data/swahili_data.csv',delimiter='\t')['URL'])
swahili_data_url[:10]

['https://onelink.to/xanf67',
 'https://bit.ly/3Nhs27X',
 'http://onelink.to/xanf67',
 'https://onelink.to/xanf67',
 'https://onelink.to/xanf67',
 'https://onelink.to/xanf67',
 'https://bit.ly/M-PesaTC',
 'https://onelink.to/xanf67',
 'https://onelink.to/xanf67',
 'https://onelink.to/xanf67']

In [9]:
import re

def find_first_slash_preceded_by_number(s):
    # Regular expression to find the first instance of a number followed by '/'
    match = re.search(r'\d+/', s)
    
    if match:
        return match.start() + len(match.group()) - 1  # Return the index of '/'
    else:
        return -1  # Return -1 if no match is found

# Example usage
string = "example77/test 88/test2 99/test3"
index = find_first_slash_preceded_by_number(string)
print(index)  # Outputs the index of the first '/' preceded by a number

9


In [10]:
unique_swahili_data_url = set(swahili_data_url )

In [11]:
for idx,i in enumerate(unique_swahili_data_url):
    if '..' in i:
        if 'www' in i:
            unique_swahili_data_url[idx] = ''
        else:
            unique_swahili_data_url[idx] = unique_swahili_data_url[idx].replace('..','.')

In [12]:
unique_swahili_data_url = [i for i in unique_swahili_data_url if i!='']

In [14]:
swahili_data_dataset = {'ham':list(pd.read_csv('../URL Data/swahili_data_ham.csv',delimiter='\t')['URL']),'spam':list(pd.read_csv('../URL Data/swahili_data_spam.csv',delimiter='\t')['URL'])}

In [15]:
swahili_data_url_in_ham = [0]*len(unique_swahili_data_url)
swahili_data_url_in_spam = [0]*len(unique_swahili_data_url)

for idx,i in enumerate(unique_swahili_data_url):
    for j in swahili_data_dataset['ham']:
        if not isinstance(j, str):
            continue
        if i in j:
            swahili_data_url_in_ham[idx] = 1
            break
    for j in swahili_data_dataset['spam']:
        if not isinstance(j, str):
            continue
        if i in j:
            swahili_data_url_in_spam[idx] = 1
            break

print(swahili_data_url_in_ham.count(1))
print(swahili_data_url_in_spam.count(1))

9
0


In [16]:
common_urls = []
for i in range(len(swahili_data_url_in_ham)):
    if swahili_data_url_in_ham[i]==swahili_data_url_in_spam[i]:
        common_urls.append(unique_swahili_data_url[i])

len(common_urls)

0

In [17]:
swahili_data_dataset['Unique Url'] = unique_swahili_data_url

In [18]:
import tldextract

def FQDN(Url):
    
    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [19]:
swahili_data_dataset['FQDN'] = [FQDN(i) for i in swahili_data_dataset['Unique Url']]

len(set(swahili_data_dataset['FQDN']))

5

In [20]:
a = soupFromUrl('https://facebook.com')

print(a)

[77867, 613, 200, 0]


The below query last ran on 5 October 2025

In [21]:
website_size, text_content_length, status_code, parked = [],[],[],[]

for i in swahili_data_dataset['FQDN']:
    a = soupFromUrl('https://'+i)

    website_size.append(a[0])
    text_content_length.append(a[1])
    status_code.append(a[2])
    parked.append(a[3])


# parked = [0]*len(swahili_data_dataset)

# for idx,i in enumerate(swahili_data_dataset['FQDN']):
#     if swahili_data_dataset['Status Code'][idx]==200:
#         a = soupFromUrl('https://'+i)
#         parked[idx] = a[3]
#         # break

# print(parked)

In [22]:
swahili_data_dataset['Website Size in KB'] = website_size
swahili_data_dataset['Website Textual Content Length'] = text_content_length
swahili_data_dataset['Status Code'] = status_code

swahili_data_dataset['Parked'] = parked

In [23]:
for i in swahili_data_dataset:
    print(len(swahili_data_dataset[i]))

31
0
9
9
9
9
9
9


In [24]:
swahili_data_dataset['ham'] = swahili_data_url_in_ham
swahili_data_dataset['spam'] = swahili_data_url_in_spam

In [25]:
swahili_data_dataset

{'ham': [1, 1, 1, 1, 1, 1, 1, 1, 1],
 'spam': [0, 0, 0, 0, 0, 0, 0, 0, 0],
 'Unique Url': ['www.zap.co.mz',
  'http://onelink.to/xanf67',
  'https://onelink.to/xanf67',
  'http://movtv.co.mz',
  'https://bit.ly/M-PesaTC',
  'https://bit.ly/2Wbzubn',
  'https://bit.ly/3Nhs27X',
  'https://bit.ly/3Lt7QAy',
  'https://yabadoo.tv/client/tmcel'],
 'FQDN': ['www.zap.co.mz',
  'onelink.to',
  'onelink.to',
  'movtv.co.mz',
  'bit.ly',
  'bit.ly',
  'bit.ly',
  'bit.ly',
  'yabadoo.tv'],
 'Website Size in KB': [7707,
  301512,
  301512,
  0,
  137740,
  137740,
  137740,
  137740,
  31743],
 'Website Textual Content Length': [10,
  2288,
  2288,
  0,
  10189,
  10189,
  10189,
  10189,
  1696],
 'Status Code': [200, 200, 200, -1, 200, 200, 200, 200, 200],
 'Parked': [0, 0, 0, 0, 1, 1, 1, 1, 0]}

In [26]:
# swahili_data_dataset = pd.read_csv('../Dataset/URL Data/swahili_data Websites Analysis.csv')
swahili_data_dataset = pd.DataFrame.from_dict(swahili_data_dataset)
swahili_data_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,1,0,www.zap.co.mz,www.zap.co.mz,7707,10,200,0
1,1,0,http://onelink.to/xanf67,onelink.to,301512,2288,200,0
2,1,0,https://onelink.to/xanf67,onelink.to,301512,2288,200,0
3,1,0,http://movtv.co.mz,movtv.co.mz,0,0,-1,0
4,1,0,https://bit.ly/M-PesaTC,bit.ly,137740,10189,200,1


In [27]:
swahili_data_dataset['Status Code'].value_counts()

Status Code
 200    8
-1      1
Name: count, dtype: int64

In [28]:
numbers_to_replace = [501,403, 401]

# Value to replace with
new_value = 200

# Update the column
swahili_data_dataset.loc[swahili_data_dataset['Status Code'].isin(numbers_to_replace), 'Status Code'] = new_value

In [29]:
swahili_data_dataset['Status Code'].value_counts()

Status Code
 200    8
-1      1
Name: count, dtype: int64

In [30]:
print(len(swahili_data_dataset[(swahili_data_dataset['Status Code']==200) & (swahili_data_dataset['ham']==1)]))
print(len(swahili_data_dataset[(swahili_data_dataset['Status Code']==200) & (swahili_data_dataset['spam']==1)]))

8
0


In [31]:
swahili_data_dataset['Parked'].value_counts()

Parked
0    5
1    4
Name: count, dtype: int64

In [32]:
print(len(swahili_data_dataset[(swahili_data_dataset['Parked']==1) & (swahili_data_dataset['ham']==1)]))
print(len(swahili_data_dataset[(swahili_data_dataset['Parked']==1) & (swahili_data_dataset['ham']==0)]))

4
0


In [33]:
swahili_data_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,1,0,www.zap.co.mz,www.zap.co.mz,7707,10,200,0
1,1,0,http://onelink.to/xanf67,onelink.to,301512,2288,200,0
2,1,0,https://onelink.to/xanf67,onelink.to,301512,2288,200,0
3,1,0,http://movtv.co.mz,movtv.co.mz,0,0,-1,0
4,1,0,https://bit.ly/M-PesaTC,bit.ly,137740,10189,200,1


In [34]:
swahili_data_dataset.to_csv('../URL Data/swahili_data Websites Analysis.csv', index=None)

In [35]:
len(swahili_data_dataset)

9